# Connection Check & Catalog Discovery

This notebook:
1. Confirms AWS credentials work.
2. Lists the Glue databases and tables (so we see the *current* names).
3. Pulls a tiny sample from the dimension tables.
4. Locates the live telemetry (fact) table and peeks at the compliance results.

**Before running:** make sure you have logged in and selected the profile:

```powershell
aws sso login --profile ciccada
$env:AWS_PROFILE = "ciccada"
```

## 1. Credentials check

In [1]:
import boto3
session = boto3.Session(profile_name="ciccada", region_name="ap-southeast-2")
ident = session.client("sts").get_caller_identity()
print("Account:", ident["Account"])
print("Identity:", ident["Arn"])
# If this errors with 'Unable to locate credentials', run `aws sso login` and
# set AWS_PROFILE, then restart the kernel.

Account: 130340360668
Identity: arn:aws:sts::130340360668:assumed-role/AWSReservedSSO_AWSAdministratorAccess_58ece215f84a4b54/z3553082_sa@ad.unsw.edu.au


## 2. The catalog
Mapping (`SolA_ts4`, `SolA_circuits`) to whatever they are called today.

In [4]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../shared").resolve()))

from aws_config import aq, databases, tables

databases()

,Database,Description
0,bom_nci,
1,default,Default Hive database
2,elb_logdb,
3,sapn2022,
4,solar_analytics,Migrated from Hive Metastore
5,solar_analytics_iceberg,
6,test_db,
7,type_probe,


In [5]:
# Tables in the main analytics database
tables("solar_analytics")[["Database", "Table", "TableType"]]

,Database,Table,TableType
0,solar_analytics,circuits,EXTERNAL_TABLE
1,solar_analytics,compliance_voltvar,EXTERNAL_TABLE
2,solar_analytics,compliance_voltwatt,EXTERNAL_TABLE
3,solar_analytics,meta_single_inverters,EXTERNAL_TABLE
4,solar_analytics,meta_single_inverters_wrong_capacity,EXTERNAL_TABLE
5,solar_analytics,meta_single_inverters_wrong_capacity_up2_3c,EXTERNAL_TABLE
6,solar_analytics,partition_lookup,EXTERNAL_TABLE
7,solar_analytics,raw_bom_2024_6,EXTERNAL_TABLE
8,solar_analytics,sites,EXTERNAL_TABLE
9,solar_analytics,test_sola_2025_12,EXTERNAL_TABLE


In [6]:
# The Iceberg database
tables("solar_analytics_iceberg")[["Database", "Table", "TableType"]]

,Database,Table,TableType
0,solar_analytics_iceberg,all_uncurtailedpv,EXTERNAL_TABLE
1,solar_analytics_iceberg,all_uncurtailedpv_v2,EXTERNAL_TABLE
2,solar_analytics_iceberg,circuits,EXTERNAL_TABLE
3,solar_analytics_iceberg,conformance_antiisland,EXTERNAL_TABLE
4,solar_analytics_iceberg,conformance_sust_op,EXTERNAL_TABLE
5,solar_analytics_iceberg,conformance_sust_op_3w,EXTERNAL_TABLE
6,solar_analytics_iceberg,conformance_voltvar,EXTERNAL_TABLE
7,solar_analytics_iceberg,conformance_voltvar_v2,EXTERNAL_TABLE
8,solar_analytics_iceberg,conformance_voltwatt,EXTERNAL_TABLE
9,solar_analytics_iceberg,conformance_voltwatt_v2,EXTERNAL_TABLE


## 3. Sample the dimension tables

In [7]:
aq("SELECT * FROM circuits LIMIT 5")

,site_id,device_id,circuit_id,device_type,circuit_polarity,circuit_type,is_pv
0,484720983,135961,84703,Watt Watcher,1,pv_site_net,True
1,484720983,135961,84704,Watt Watcher,1,pv_site_net,True
2,484720983,135961,84705,Watt Watcher,1,pv_site_net,True
3,150652475,140483,658777,Watt Watcher,1,ac_load_net,False
4,150652475,140483,658778,Watt Watcher,1,ac_load_net,False


In [8]:
aq("SELECT * FROM sites LIMIT 5")

,site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,inverter_count,pv_install_date,manufacturer,model,ac_capacity_kw_exploaded,installed_after_18_dec_2021
0,1944472430,NSW,2502.0,150.85,-34.470,Endeavour,5.18,5.0,3.0,2021-08-09,1.0,2021-08-09,Sungrow,SG5KTL,5.0,False
1,1245528685,QLD,4078.0,152.95,-27.630,Energex,15.00,10.0,6.8,2024-09-18,1.0,2024-09-18,Generic Inverter,10.0kW,10.0,True
2,1651877625,NSW,2350.0,151.70,-30.510,Essential,11.55,9.6,5.0,2019-06-27,2.0,2018-11-02,Generic Inverter,5kW,5.0,False
3,1555410224,QLD,4814.0,146.75,-19.305,Ergon,13.28,10.0,5.0,2024-03-07,1.0,2024-03-06,Sungrow,SG10RS-ADA,10.0,True
4,623277618,QLD,4211.0,153.30,-27.990,Energex,15.75,10.0,5.0,2024-03-18,1.0,2023-06-30,Fronius,Primo GEN24 10.0,10.0,True


In [9]:
# The partition lookup tells you which (year, month) partitions actually exist
aq("SELECT * FROM partition_lookup LIMIT 20")

,year,month,is_pv
0,2024,1,False
1,2024,1,True
2,2024,10,False
3,2024,10,True
4,2024,11,False
5,2024,11,True
6,2024,12,False
7,2024,12,True
8,2024,2,False
9,2024,2,True


## 4. Find the live telemetry table

In [10]:
"""# partition-pruned sample
YEAR = 2025
MONTH = 1
sample = aq(f'''
    SELECT circuit_id, t_stamp, voltage, power, energy_reactive
    FROM {FACT_TABLE}
    WHERE is_pv = True AND year = {YEAR} AND month = {MONTH}
      AND circuit_id = 547781
    ORDER BY t_stamp
    LIMIT 20
''', database=FACT_DB)
sample"""

"# partition-pruned sample\nYEAR = 2025\nMONTH = 1\nsample = aq(f'''\n    SELECT circuit_id, t_stamp, voltage, power, energy_reactive\n    FROM {FACT_TABLE}\n    WHERE is_pv = True AND year = {YEAR} AND month = {MONTH}\n      AND circuit_id = 547781\n    ORDER BY t_stamp\n    LIMIT 20\n''', database=FACT_DB)\nsample"

## 5. Analysis outputs

These tables are the *results* the report was built from. Their schemas tell us
exactly what the `SolA2024_Analysis` notebooks ultimately produced.

In [11]:
aq("SELECT * FROM compliance_voltwatt LIMIT 5")

,site_id,s_id,year,month,day,noncompliance_voltwatt_count,noncompliance_voltwatt_sum,total_count
0,755163749,S5569,2025,4,28,67,315.360426,80
1,1594693506,S11739,2025,1,13,95,254.730175,274
2,755163749,S5569,2024,8,18,77,410.032359,94
3,465008538,S3443,2024,5,6,76,262.625491,267
4,465008538,S3443,2025,3,5,81,266.511457,127


In [12]:
# What inverter metadata was inferred (nameplate capacity, etc.)
aq("SELECT * FROM meta_single_inverters LIMIT 5")

,circuit_id,site_id,device_id,device_type,circuit_polarity,circuit_type,is_pv,state,postcode,longitude,...,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,inverter_count,pv_install_date,manufacturer,model,ac_capacity_kw_exploaded,installed_after_18_dec_2021
0,5699,2035522732,175956,Watt Watcher,-1,pv_site_net,True,QLD,4211.0,153.35,...,30.00,25.0,NaN,2015-09-04,1.0,2015-09-03,SMA,Sunny Tripower STP25000 TL-30,25.0,False
1,5751,1763489963,180346,Watt Watcher,-1,pv_site_net,True,NSW,2000.0,151.20,...,2.40,3.0,NaN,2015-08-24,1.0,2015-01-01,SolarEdge,SE3000,3.0,False
2,8631,1108052410,166629,Watt Watcher,1,pv_site_net,True,VIC,3977.0,145.25,...,4.16,5.0,NaN,2015-10-30,1.0,2015-10-22,Zeversolar,Evershine TL5000,5.0,False
3,14260,623711824,195310,Watt Watcher,1,pv_site_net,True,NSW,2620.0,149.25,...,5.00,4.6,NaN,2016-02-03,1.0,2016-02-02,SMA,Sunny Boy SB 5000 TL-20,4.6,False
4,14369,1625427199,89086,Watt Watcher,1,pv_site_net,True,NSW,2830.0,148.55,...,10.08,10.0,NaN,2015-12-04,1.0,2015-12-02,ABB,PVI-10.0-TL-OUTD,10.0,False
